# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgrimJain-quantum/internship-remote/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [26]:
# Setup — HF auth + duckdb over the warehouse. Run this first, every session.
!pip -q install duckdb huggingface_hub

import duckdb, os, pandas as pd
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"

DIM_CLIENTS   = f"{BASE}/dim_clients.parquet"
DIM_CONTENT   = f"{BASE}/dim_content.parquet"
DAILY_MONTH   = lambda m: f"{BASE}/fact_content_daily_performance/month={m}/data_0.parquet"
QUERY_90D     = f"{BASE}/fact_content_query_90d.parquet"       # flat file — single window, Apr-Jun 2026, sealed. Mechanics only.
SEALED_SAMPLE = f"{BASE}/fact_content_daily_performance_sample.parquet"  # June 2026 — sealed test month

# My lane: fact_content_daily_performance. Real partitions run 2025-01 .. 2026-06.
# Two-month design avoids leakage by construction: features from PREV_MONTH, label from DEV_MONTH.
PREV_MONTH = "2026-02"   # feature month
DEV_MONTH  = "2026-03"   # label month, mid-panel
SEALED_MONTH = "2026-06" # never touch for label/feature development
print("Feature month:", PREV_MONTH, "| Label month:", DEV_MONTH, "| Sealed month (do not touch):", SEALED_MONTH)

con.sql(f"SELECT COUNT(*) FROM read_parquet('{DIM_CLIENTS}')").show()


Feature month: 2026-02 | Label month: 2026-03 | Sealed month (do not touch): 2026-06
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          104 │
└──────────────┘



## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My lane:** Search Intelligence → `fact_content_query_90d`.

**Unit of analysis:** one row = one search query's performance for one content item for one
client — grain is `client_id × content_id × query_hash`. It is **not** client×content:
the same content item has many rows, one per distinct query it ranks for.

**Time window:** each row already carries a fixed **trailing 90-day window** baked into it
(the table's own design, per `skills/README.md`). I read the partition `month=2026-03` — a
mid-panel month — as my dev slice, because `dim_clients.gsc_data_start` varies wildly per
client and a global calendar window would silently exclude newer clients. I never develop
label logic on `fact_content_query_90d_sample` / the June 2026 partition, since that is the
panel's last month and is reserved as a sealed test month.

**What I'd predict/rank (label or proxy):** rank each content item's query rows by search
performance — proxy label `is_top_half = 1` if `gsc_avg_position` is in the client's better
(numerically lower) half for that month, else `0`. This stays fully inside one month, so it
needs no future outcome window.

**One thing I deliberately exclude:** `avg_position = 0` rows. Per the data dictionary this
code means "no data," not rank zero — treating it as a real position would fabricate a
best-possible rank for content that was never actually measured.

In [27]:
# Quick peek at what a "row" looks like in fact_content_daily_performance for the label month.
con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, gsc_data_available,
       gsc_impressions, gsc_clicks, gsc_avg_position, gsc_sum_position
FROM read_parquet('{DAILY_MONTH(DEV_MONTH)}')
LIMIT 5
""").show()


┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────────┬─────────────────┬────────────┬───────────────────┬──────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_data_available │ gsc_impressions │ gsc_clicks │ gsc_avg_position  │ gsc_sum_position │
│    date     │         varchar         │         varchar          │      boolean       │      int64      │   int64    │      double       │      int64       │
├─────────────┼─────────────────────────┼──────────────────────────┼────────────────────┼─────────────────┼────────────┼───────────────────┼──────────────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │ true               │              20 │          0 │              3.35 │               67 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │ true               │               1 │          0 │               0.0 │                0 │
│ 2026-03-01  │ client_73cda7b4e4f265ea 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
I ran `DESCRIBE` (cell below) to get the real column names before finalizing this table —
filling in the exact names after first run. Structure, from the docs + dictionary:

| Bucket | Fields (confirm exact names from DESCRIBE output) | Why |
|---|---|---|
| **Feature** | query-level metrics for the 90-day window: impressions, clicks, CTR, `gsc_avg_position` (excluding the `=0` rows), plus content-level context pulled in with `ANY_VALUE()` — `content_type`, `word_count` (with a `has_word_count` flag rather than `fillna(0)`, since missingness follows `content_type`) | All knowable at the moment the 90-day window closes — before any future-period decision |
| **Label / proxy** | `is_top_half` (derived from `gsc_avg_position` within `client_id`, this month only) | The thing I'm ranking by; never also used as a feature |
| **Context** | `client_id`, `content_id`, `query_hash` | Pseudonyms — for joining/grouping/splitting (grouped train/test split by `client_id`) only, never fed to a model |
| **Excluded** | any `trend_pct` / `trend_direction` / `is_declining_label`-style column if present in a joined table, and any GA4 column where `ga4_data_available = FALSE` | These are label-derived or zero-filled-as-placeholder, not real signal — including them is exactly the leakage trap I run in the next section |


In [28]:
# Real schema — already confirmed via DESCRIBE screenshot (kept here for a reproducible run-all).
print("--- fact_content_daily_performance ---")
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DAILY_MONTH(DEV_MONTH)}')").show(max_rows=100)

print("--- dim_content ---")
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DIM_CONTENT}')").show(max_rows=100)

print("--- dim_clients ---")
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DIM_CLIENTS}')").show(max_rows=100)


--- fact_content_daily_performance ---
┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date              │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions          │ BIGINT      │ YES  

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
Three claims, three queries, in order:
1. **Grain** — `client_id × content_id × query_hash` really is unique.
2. **Counts + date span** — how many rows this dev month, and the panel dates they cover.
3. **Availability** — filter with `IS TRUE` (`ga4_data_available`) and report survivors vs total.

In [29]:
# (1) Grain probe — should return ZERO rows if report_date x client_hash_id x content_hash_id
# is the real grain in the label month.
grain_check = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('{DAILY_MONTH(DEV_MONTH)}')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()
print("Grain violations found:", len(grain_check))
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations found: 0


,report_date,client_hash_id,content_hash_id,c


In [30]:
# (2) Row count + date span for the label month partition.
con.sql(f"""
SELECT
  COUNT(*)                          AS n_rows,
  COUNT(DISTINCT client_hash_id)    AS n_clients,
  COUNT(DISTINCT content_hash_id)   AS n_content,
  MIN(report_date)                  AS min_date,
  MAX(report_date)                  AS max_date
FROM read_parquet('{DAILY_MONTH(DEV_MONTH)}')
""").show()


┌─────────┬───────────┬───────────┬────────────┬────────────┐
│ n_rows  │ n_clients │ n_content │  min_date  │  max_date  │
│  int64  │   int64   │   int64   │    date    │    date    │
├─────────┼───────────┼───────────┼────────────┼────────────┤
│ 9841378 │        55 │    331437 │ 2026-03-01 │ 2026-03-31 │
└─────────┴───────────┴───────────┴────────────┴────────────┘



In [31]:
# (3) Availability — filter with IS TRUE on gsc_data_available, report survivors vs total.
con.sql(f"""
SELECT
  COUNT(*)                                                       AS total_rows,
  SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)    AS survives_gsc_filter,
  ROUND(100.0 * SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_survives
FROM read_parquet('{DAILY_MONTH(DEV_MONTH)}')
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┬──────────────┐
│ total_rows │ survives_gsc_filter │ pct_survives │
│   int64    │       int128        │    double    │
├────────────┼─────────────────────┼──────────────┤
│    9841378 │             3611061 │         36.7 │
└────────────┴─────────────────────┴──────────────┘



## 3b. Five features + the leakage trap

*Build a small feature frame for the dev month, one "knowable at the decision moment
because…" line per feature. Then add ONE label-derived column on purpose, watch the score
jump toward perfect, delete it, keep the honest number.*

**Five features**, all from the `_prev30` window — strictly before the label window:
1. `impressions_prev30` — closed out before the last-30-day label window even opens.
2. `clicks_prev30` — same.
3. `ctr_prev30` (computed from the two above) — same window, so still safe.
4. `word_count` (+ `has_word_count` flag; `dim_content`) — set at content creation/last edit,
   long before this window.
5. `content_type` (`dim_content`) — categorical, set at publish time.

**Label (proxy):** `is_top_half`, derived from `avg_position_last30` within `client_hash_id`
for the dev month — the thing being predicted, never a feature.

**The trap:** I add `position_pct_rank`, computed directly from `avg_position_last30` — i.e.
directly from the label. That's the leak from notebook 02, reproduced on real warehouse data.


In [32]:
# Feature month (February): aggregate to one row per content item.
feat = con.sql(f"""
SELECT
  client_hash_id,
  content_hash_id,
  SUM(gsc_impressions) AS impressions_prevmonth,
  SUM(gsc_clicks)       AS clicks_prevmonth
FROM read_parquet('{DAILY_MONTH(PREV_MONTH)}')
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()
feat["ctr_prevmonth"] = feat["clicks_prevmonth"] / feat["impressions_prevmonth"].replace(0, pd.NA)
feat["ctr_prevmonth"] = feat["ctr_prevmonth"].fillna(0)

# Label month (March): weighted average position, correctly aggregated.
label = con.sql(f"""
SELECT
  client_hash_id,
  content_hash_id,
  SUM(gsc_sum_position) AS sum_position,
  SUM(gsc_impressions)  AS impressions_labelmonth
FROM read_parquet('{DAILY_MONTH(DEV_MONTH)}')
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) > 0
""").df()
label["weighted_avg_position"] = label["sum_position"] / label["impressions_labelmonth"]

# Content-level context.
content = con.sql(f"""
SELECT content_hash_id, word_count, content_type
FROM read_parquet('{DIM_CONTENT}')
WHERE is_deleted IS NOT TRUE
""").df()

# Join: Feb features + March label + content context. Inner join by design (see limitation above).
df = feat.merge(label[["client_hash_id", "content_hash_id", "weighted_avg_position"]],
                 on=["client_hash_id", "content_hash_id"], how="inner")
df = df.merge(content, on="content_hash_id", how="left")

df["has_word_count"] = df["word_count"].notna().astype(int)
df["word_count"] = df["word_count"].fillna(0)

# Proxy label: top half of March weighted avg position, within each client.
df["is_top_half"] = (
    df.groupby("client_hash_id")["weighted_avg_position"]
      .rank(pct=True) <= 0.5
).astype(int)

print(df.shape)
df.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(134238, 10)


,client_hash_id,content_hash_id,impressions_prevmonth,clicks_prevmonth,ctr_prevmonth,weighted_avg_position,word_count,content_type,has_word_count,is_top_half
0,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,0.200000,7.000000,957,feedly article,1,0
1,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,0.000000,7.372093,861,feedly article,1,0
2,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,0.000000,5.760784,823,feedly article,1,0
3,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,0.000000,7.000000,786,feedly article,1,0
4,client_3ffa76342f366962,content_0674cc4ae0f68a90,74.0,1.0,0.013514,6.363636,918,feedly article,1,0


In [33]:
# Honest quick score — features only, no leak.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

feature_cols = ["impressions_prevmonth", "clicks_prevmonth", "ctr_prevmonth", "word_count", "has_word_count", "content_type"]
X = df[feature_cols]
y = df["is_top_half"]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["content_type"]),
], remainder="passthrough")

pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])

honest_score = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc").mean()
print(f"Honest AUC (no leak): {honest_score:.3f}")


Honest AUC (no leak): 0.634


In [34]:

# The trap: add a column derived directly from the label and watch the score jump.
df["position_pct_rank"] = df.groupby("client_hash_id")["weighted_avg_position"].rank(pct=True)  # == the label, restated

leak_cols = feature_cols + ["position_pct_rank"]
X_leak = df[leak_cols]

pre_leak = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["content_type"]),
], remainder="passthrough")
pipe_leak = Pipeline([("pre", pre_leak), ("clf", LogisticRegression(max_iter=1000))])

leaked_score = cross_val_score(pipe_leak, X_leak, y, cv=5, scoring="roc_auc").mean()
print(f"Leaked AUC (with position_pct_rank): {leaked_score:.3f}")
print(f"Jump: {leaked_score - honest_score:+.3f}")

# Delete the leak and keep the honest number.
del df["position_pct_rank"]
print(f"\nKeeping honest AUC = {honest_score:.3f} as the real number for this contract.")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Leaked AUC (with position_pct_rank): 1.000
Jump: +0.366

Keeping honest AUC = 0.634 as the real number for this contract.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Unbalanced history.** `dim_clients.gsc_data_start` differs per client, so a global
  calendar month mixes clients with years of history and clients with weeks. This contract
  does not correct for that — it only checks that the dev month has usable rows per client.
- **GSC-only early rows.** Rows before a client's `ga4_data_start` have GA4 columns
  zero-filled with `ga4_data_available = FALSE`. Section 3's availability query shows how
  much of the month survives an `IS TRUE` filter; the rest is GSC-only signal at best.
- **Window overlap.** `fact_content_query_90d`'s 90-day window overlaps the last months of
  `fact_content_daily_performance`. If a future label is ever built from the daily table,
  only `*_prev30`-style columns from the query table would be safe features — this contract's
  proxy label avoids the issue entirely by staying inside one closed 90-day window, but any
  extension of this work that predicts a *future* outcome must re-check window alignment
  before reusing these features.
- **What this data can't tell you at all:** intent behind a query, why a position moved,
  or anything about clients outside the ~two-thirds with usable search/analytics history —
  the other third have little to no data and were implicitly dropped by the `IS TRUE` filter.

In [35]:
# Named-limitation check, quantified: how many clients have any usable (gsc_data_available)
# rows in the label month vs how many exist in dim_clients at all.
con.sql(f"""
WITH usable AS (
  SELECT DISTINCT client_hash_id
  FROM read_parquet('{DAILY_MONTH(DEV_MONTH)}')
  WHERE gsc_data_available IS TRUE
)
SELECT
  (SELECT COUNT(*) FROM read_parquet('{DIM_CLIENTS}')) AS n_clients_total,
  (SELECT COUNT(*) FROM usable)                         AS n_clients_usable_this_month
""").show()


┌─────────────────┬─────────────────────────────┐
│ n_clients_total │ n_clients_usable_this_month │
│      int64      │            int64            │
├─────────────────┼─────────────────────────────┤
│             104 │                          47 │
└─────────────────┴─────────────────────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.